# machine-learning_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (117).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 1 |
| Detected functions | create_advanced_features, objective |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
# --- 1. Install Necessary Packages ---
print("Installing required packages: lightgbm, catboost, imbalanced-learn, and optuna...")
!pip install -q lightgbm catboost imbalanced-learn optuna
print("Installation complete.")

# --- 2. Import Libraries ---
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgb
import catboost as ctb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# --- 3. Configuration ---
warnings.filterwarnings('ignore')
SEED = 42
N_SPLITS = 5
OPTUNA_TRIALS = 30

# --- 4. Load Data ---
print("\nLoading data...")
try:
    train_df = pd.read_parquet('train.parquet')
    test_df = pd.read_parquet('test.parquet')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Make sure train.parquet and test.parquet are uploaded to the Colab environment.")
    exit()

# --- 5. Advanced Feature Engineering ---
def create_advanced_features(df):
    """Creates a rich set of time-series and interaction features from a dataframe."""
    df_copy = df.copy()

    # Basic Date Features
    df_copy['hour'] = df_copy['Date'].dt.hour
    df_copy['weekday'] = df_copy['Date'].dt.weekday
    df_copy['dayofyear'] = df_copy['Date'].dt.dayofyear
    df_copy['month'] = df_copy['Date'].dt.month

    # Cyclical Features
    df_copy['hour_sin'] = np.sin(2 * np.pi * df_copy['hour']/24)
    df_copy['hour_cos'] = np.cos(2 * np.pi * df_copy['hour']/24)
    df_copy['weekday_sin'] = np.sin(2 * np.pi * df_copy['weekday']/7)
    df_copy['weekday_cos'] = np.cos(2 * np.pi * df_copy['weekday']/7)

    # Sort by date to create rolling/lag features correctly
    df_copy = df_copy.sort_values('Date')

    features_to_process = [f'X{i}' for i in range(1, 6)]
    windows = [3, 6, 12, 24] # Representing hours

    for col in features_to_process:
        # Lag features
        for lag in [1, 2, 3]:
            df_copy[f'{col}lag{lag}'] = df_copy[col].shift(lag)

        # Rolling features
        for w in windows:
            rolling_window = df_copy[col].rolling(window=w)
            df_copy[f'{col}roll_mean{w}'] = rolling_window.mean()
            df_copy[f'{col}roll_std{w}'] = rolling_window.std()
            df_copy[f'{col}roll_max{w}'] = rolling_window.max()
            df_copy[f'{col}roll_min{w}'] = rolling_window.min()

    # Drop the original Date column as it has been used
    return df_copy.drop('Date', axis=1)

print("\nPerforming advanced feature engineering...")

# Store identifiers and target before combining
train_len = len(train_df)
test_ids = test_df['ID']
y = train_df['target'].astype(int)

# Combine for consistent feature creation, dropping unneeded columns first
train_drop = train_df.drop(['target'], axis=1)
test_drop = test_df.drop(['ID'], axis=1)
combined_df = pd.concat([train_drop, test_drop], ignore_index=True)

combined_featured = create_advanced_features(combined_df)

# Fill NaNs created by rolling/lag features
for col in combined_featured.columns[combined_featured.isna().any()].tolist():
    combined_featured[col] = combined_featured[col].fillna(combined_featured[col].median())

# Split back into train and test
X = combined_featured.iloc[:train_len]
X_test = combined_featured.iloc[train_len:]

print("Feature engineering complete.")
print(f"Number of features created: {X.shape[1]}")

# --- 6. Hyperparameter Tuning (Optuna) ---
def objective(trial, X_data, y_data, model_type):
    if model_type == 'lgbm':
        params = {
            'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'boosting_type': 'gbdt', 'n_estimators': 1000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'max_depth': trial.suggest_int('max_depth', 5, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'random_state': SEED, 'n_jobs': -1
        }
        model_class = lgb.LGBMClassifier
    elif model_type == 'catboost':
        params = {
            'objective': 'Logloss', 'eval_metric': 'AUC', 'iterations': 1000, 'verbose': 0,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 1e-9, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_seed': SEED
        }
        model_class = ctb.CatBoostClassifier

    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=SEED)),
        ('model', model_class(**params))
    ])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, val_idx in skf.split(X_data, y_data):
        X_train, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
        y_train, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, preds))
    return np.mean(scores)

print("\nTuning LightGBM...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(lambda trial: objective(trial, X, y, 'lgbm'), n_trials=OPTUNA_TRIALS, timeout=900)
lgb_best_params = study_lgbm.best_params

print("\nTuning CatBoost...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(lambda trial: objective(trial, X, y, 'catboost'), n_trials=OPTUNA_TRIALS, timeout=900)
cat_best_params = study_cat.best_params

# --- 7. Ensemble Model Training ---
lgbm_full_params = {**lgb_best_params, 'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'boosting_type': 'gbdt', 'n_estimators': 2000, 'random_state': SEED, 'n_jobs': -1}
catboost_full_params = {**cat_best_params, 'objective': 'Logloss', 'eval_metric': 'AUC', 'iterations': 2000, 'verbose': 0, 'random_seed': SEED}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
final_test_preds = np.zeros(len(X_test))

print(f"\nStarting ensemble training with {N_SPLITS}-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"--- Fold {fold+1}/{N_SPLITS} ---")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    smote = SMOTE(random_state=SEED)
    X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

    # Train LightGBM
    lgbm = lgb.LGBMClassifier(**lgbm_full_params)
    lgbm.fit(X_resampled, y_resampled, eval_set=[(X_val_scaled, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])

    # Train CatBoost
    cat = ctb.CatBoostClassifier(**catboost_full_params)
    cat.fit(X_resampled, y_resampled, eval_set=[(X_val_scaled, y_val)], early_stopping_rounds=100, verbose=0)

    # Predict and average for ensemble
    lgbm_preds = lgbm.predict_proba(X_val_scaled)[:, 1]
    cat_preds = cat.predict_proba(X_val_scaled)[:, 1]
    oof_preds[val_idx] = 0.5 * lgbm_preds + 0.5 * cat_preds

    lgbm_test_preds = lgbm.predict_proba(X_test_scaled)[:, 1]
    cat_test_preds = cat.predict_proba(X_test_scaled)[:, 1]
    final_test_preds += (0.5 * lgbm_test_preds + 0.5 * cat_test_preds) / N_SPLITS

print("Model training complete.")

# --- 8. Threshold Optimization and Submission ---
thresholds = np.linspace(0, 1, 100)
f1_scores_list = [f1_score(y, oof_preds > t) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores_list)]
print(f"\nBest threshold for ensemble found at: {best_threshold:.4f}")
print(f"Ensemble OOF ROC AUC score: {roc_auc_score(y, oof_preds):.4f}")
print(f"Ensemble OOF F1 score: {np.max(f1_scores_list):.4f}")

final_test_labels = (final_test_preds > best_threshold).astype(int)

# --- Create Submission File ---
print("\nGenerating submission file...")
submission_df = pd.DataFrame({'ID': test_ids, 'target': final_test_labels})
submission_df['target'] = submission_df['target'].astype(str)
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' has been generated successfully!")
print("Final submission file head:")
print(submission_df.head())